<a href="https://colab.research.google.com/github/raviklog/ClustersTest/blob/master/PythonCore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Traditional loop vs List Comprehension
raw_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# High-performance, low-overhead inline looping
squared_evens = [x**2 for x in raw_data if x % 2 == 0]

print(f"Processed Array/List: {squared_evens}")
# Output: [4, 16, 36, 64, 100]


Processed Array/List: [4, 16, 36, 64, 100]


In [ ]:
def process_http_response(status_code):
    match status_code:
        case 200 | 201:
            return "Success"
        case 400 | 404 as error if error == 404:
            return "Not Found Error"
        case 500:
            return "Server Error"
        case _:
            return "Unknown Status"

print(process_http_response(510))


Unknown Status


In [ ]:
def append_to_list(element, target_list=[]):
    target_list.append(element)
    return target_list

# Expectation: Three separate lists. Reality: Shared state instance reference.
print(append_to_list("A"))  # ['A']
print(append_to_list("B"))  # ['A', 'B']  <-- Bug! The definition-time list persists.

# The Architect's fix:
def append_to_list_fixed(element, target_list=None):
    if target_list is None:
        target_list = []
    target_list.append(element)
    return target_list


['A']
['A', 'B']


In [ ]:
def rotate_matrix(matrix: list[list[int]]) -> None:
    """
    Rotates an NxN matrix 90 degrees clockwise in-place.
    """
    n = len(matrix)
    # Step 1: Transpose the matrix
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]

    # Step 2: Reverse each row
    for row in matrix:
        row.reverse()

# --- Explicitly Formatted Test Execution ---
demo_matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

print("Original Matrix:")
for r in demo_matrix:
    print(r)

rotate_matrix(demo_matrix)

print("\nRotated 90 Degrees Clockwise:")
for r in demo_matrix:
    print(r)


Original Matrix:
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]

Rotated 90 Degrees Clockwise:
[7, 4, 1]
[8, 5, 2]
[9, 6, 3]


In [ ]:
class DatabaseConnectionContext:
    def __init__(self, connection_string: str):
        self.conn_str = connection_string

    def __enter__(self):
        # Equivalent to allocating the resource in a C# constructor/open call
        print(f"[Resource] Opening active pool to: {self.conn_str}")
        return self # This object is bound to the target variable in the 'as' clause

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Equivalent to the C# Dispose() block.
        # This executes even if an unhandled exception occurs inside the code block.
        print("[Resource] Flushing buffers and closing active connection pool gracefully.")

        if exc_type:
            print(f"[Alert] Exception intercepted inside context: {exc_val}")
            return False # Return False to let the exception bubble up, or True to swallow it.

# --- Execution ---
with DatabaseConnectionContext("Server=EnterpriseProd;DB=Core;") as db:
    print("  Executing database transactional logic operations...")
    # Uncomment the line below to test the exception handling integration:
    # raise Exception("Database connection timeout error.")

print("Context exited completely.")


[Resource] Opening active pool to: Server=EnterpriseProd;DB=Core;
  Executing database transactional logic operations...
[Resource] Flushing buffers and closing active connection pool gracefully.
Context exited completely.


In [ ]:
import sys

# 1. List Comprehension (Eager Evaluation - allocates memory for everything immediately)
# This is equivalent to creating a C# List<int> and populating it with a loop.
eager_list = [x for x in range(1000000)]
print(f"Eager List Size in Heap: {sys.getsizeof(eager_list):,} bytes")
# Output: ~8.4 Megabytes

# 2. Generator Expression (Lazy Evaluation - allocates memory only for the state machine)
# This is equivalent to an un-executed LINQ expression or an IEnumerable method.
lazy_generator = (x for x in range(1000000))
print(f"Lazy Generator Size in Heap: {sys.getsizeof(lazy_generator):,} bytes")
# Output: ~104 bytes!


Eager List Size in Heap: 8,448,728 bytes
Lazy Generator Size in Heap: 192 bytes


In [ ]:
# Simulated high-volume enterprise log stream
raw_logs = [
    "2026-07-06 10:00:00 INFO - System heartbeat OK",
    "2026-07-06 10:01:15 WARN - Disk space at 88%",
    "2026-07-06 10:02:30 ERROR - Database query timeout occurred",
    "2026-07-06 10:03:45 INFO - User session initialized",
    "2026-07-06 10:05:00 CRITICAL - Auth service connection dropped"
]

def log_reader(logs: list[str]):
    """Generator Stage 1: Streams lines one by one"""
    for line in logs:
        yield line

def error_filter(log_stream):
    """Generator Stage 2: In-line filter transformation"""
    for log in log_stream:
        if "ERROR" in log or "CRITICAL" in log:
            yield log.upper()

# --- Assembly of the Pipeline ---
# At this point, absolutely zero processing has occurred. No data has moved.
stream = log_reader(raw_logs)
pipeline = error_filter(stream)

print("Starting pipeline ingestion...")

# Data is pulled into memory only as requested by the consumer loop (Equivalent to foreach)
for alert in pipeline:
    print(f"[ALARM TRIGGERED]: {alert}")


Starting pipeline ingestion...
[ALARM TRIGGERED]: 2026-07-06 10:02:30 ERROR - DATABASE QUERY TIMEOUT OCCURRED
[ALARM TRIGGERED]: 2026-07-06 10:05:00 CRITICAL - AUTH SERVICE CONNECTION DROPPED


In [ ]:
class StepIterator:
    """A custom collection iterator that counts by a specified stride increment."""
    def __init__(self, start: int, end: int, step: int):
        self.current = start
        self.end = end
        self.step = step

    def __iter__(self):
        # The object returns itself as the iterator iterator mechanism
        return self

    def __next__(self) -> int:
        if self.current >= self.end:
            # Python uses Exceptions for standard control flow completion signals!
            raise StopIteration

        value_to_return = self.current
        self.current += self.step
        return value_to_return

# --- Execution ---
print("\nTesting Custom Iterator Protocol:")
custom_stream = StepIterator(start=10, end=50, step=10)

for val in custom_stream:
    print(f"Streamed Increment: {val}")



Testing Custom Iterator Protocol:
Streamed Increment: 10
Streamed Increment: 20
Streamed Increment: 30
Streamed Increment: 40


In [ ]:
# 1. The List runs instantly
eager_list = [print(f"Eagerly computing {x}") for x in range(3)]
# You will see the print statements execute IMMEDIATELY when you run this cell.

print("-" * 30)

# 2. The Generator sits completely idle
lazy_generator = (print(f"Lazily computing {x}") for x in range(3))
print("The generator object has been created, but notice that nothing has printed yet!")

print("-" * 30)
print("Now we explicitly pull the first item using next():")
next(lazy_generator)  # This triggers the state machine to run until the first yield/value.


Eagerly computing 0
Eagerly computing 1
Eagerly computing 2
------------------------------
The generator object has been created, but notice that nothing has printed yet!
------------------------------
Now we explicitly pull the first item using next():
Lazily computing 0


In [ ]:
def transaction_fraud_filter(velocity_limit: float):
    """
    An architectural state-machine coroutine.
    Maintains a rolling state window of transactions and handles real-time configuration changes.
    """
    print("[Engine] Initializing Real-time Fraud Pipeline...")
    total_processed = 0
    total_flagged = 0

    while True:
        # The code halts here. It yields out the current summary,
        # and waits for the infrastructure to .send() it a new transaction array/tuple.
        received_data = yield (total_processed, total_flagged)

        # Guard clause for pipeline shutdown or empty cycles
        if received_data is None:
            continue

        # Architect Trick: If a float/dict is passed, we update config on the fly!
        if isinstance(received_data, (int, float)):
            print(f"\n[CONFIG CHANGE]: Adjusting velocity limit from {velocity_limit} to {received_data}")
            velocity_limit = received_data
            continue # Cycle back to wait for the next transaction

        # Process transaction tuple: (transaction_id, amount)
        tx_id, amount = received_data
        total_processed += 1

        if amount > velocity_limit:
            print(f"[ALERT]: Transaction {tx_id} FLAGGED. Amount ${amount:,} exceeds limit of ${velocity_limit:,}")
            total_flagged += 1

# --- Driving the Co-routine System ---
# 1. Instantiate the generator
engine = transaction_fraud_filter(velocity_limit=10000.0)

# 2. Crucial Step: "Prime" the coroutine to advance it to the first yield statement
next(engine)

# 3. Simulate streaming millions of real-time transactions hitting our ingestion layer
engine.send(("TX-101", 1200.0))
engine.send(("TX-102", 15400.0)) # Exceeds limit!

# 4. Zero-Downtime Configuration Change: Send a primitive float to update the rule engine
engine.send(20000.0)

# 5. Continue streaming under the dynamically modified business rules
engine.send(("TX-103", 15400.0)) # Safe now under new 20k limits
engine.send(("TX-104", 22000.0)) # Exceeds new limit!

# Retrieve final stats from the engine state
stats = engine.send(None)
print(f"\n[Final Report] Ingested: {stats[0]} | Flagged: {stats[1]}")


[Engine] Initializing Real-time Fraud Pipeline...
[ALERT]: Transaction TX-102 FLAGGED. Amount $15,400.0 exceeds limit of $10,000.0

[CONFIG CHANGE]: Adjusting velocity limit from 10000.0 to 20000.0
[ALERT]: Transaction TX-104 FLAGGED. Amount $22,000.0 exceeds limit of $20,000.0

[Final Report] Ingested: 4 | Flagged: 2


In [ ]:
import time

# Mock generator simulating millions of streaming raw IoT/User clicks metric tracking logs
def raw_event_streamer():
    for i in range(1, 1000001): # 1 Million telemetry iterations
        # Every 5th event is a system error code simulation
        status = "ERROR" if i % 5 == 0 else "SUCCESS"
        yield {"event_id": i, "latency_ms": (i % 150) + 10, "status": status}

def infrastructure_filter(stream):
    """Stage 2 Pipeline: Drops structural noise instantly at zero-copy speeds"""
    for event in stream:
        if event["status"] == "SUCCESS":
            yield event

def sliding_window_batcher(stream, batch_size: int):
    """Stage 3 Pipeline: Batches streaming data to prevent DB IOPS thrashing"""
    batch = []
    for event in stream:
        batch.append(event)
        if len(batch) == batch_size:
            yield batch
            batch = [] # Flush and reset memory reference pointer
    if batch:
        yield batch

# --- High Volume Execution Profile ---
start_time = time.perf_counter()

# Connect the pipeline layers together structurally (No arrays are built or executed yet)
raw_source = raw_event_streamer()
filtered_source = infrastructure_filter(raw_source)
batched_pipeline = sliding_window_batcher(filtered_source, batch_size=5000)

print("Processing 1,000,000 live streaming events across the architectural layers...")

# Drive the pipeline: Fetching 5,000 fully processed items at a time
batch_count = 0
for db_ready_batch in batched_pipeline:
    batch_count += 1
    # Simulating massive chunk database bulk insertion (e.g., PostgreSQL COPY or MongoDB BulkWrite)
    if batch_count % 50 == 0:
        print(f"  [DB Sink] Bulk inserting batch #{batch_count} containing {len(db_ready_batch)} events...")

end_time = time.perf_counter()
print(f"\nSuccessfully processed and batched in: {end_time - start_time:.4f} seconds!")


Processing 1,000,000 live streaming events across the architectural layers...
  [DB Sink] Bulk inserting batch #50 containing 5000 events...
  [DB Sink] Bulk inserting batch #100 containing 5000 events...
  [DB Sink] Bulk inserting batch #150 containing 5000 events...

Successfully processed and batched in: 0.3874 seconds!


In [ ]:
import asyncio
import time
import random

async def fetch_api_payload(transaction_id: int) -> dict:
    """
    Simulates a non-blocking asynchronous network API request.
    Equivalent to a C# HttpClient network call returning a Task<Dictionary>.
    """
    # Simulate a network latency fluctuation between 0.1 and 0.5 seconds
    latency = random.uniform(0.1, 0.5)

    # CRITICAL: We use asyncio.sleep(), NOT time.sleep().
    # asyncio.sleep yields control back to the single-threaded event loop.
    await asyncio.sleep(latency)

    # Simulate an occasional transient network failure (10% chance)
    if random.random() < 0.10:
        raise ConnectionError(f"Gateway Timeout on TX-{transaction_id}")

    return {"tx_id": transaction_id, "status": "PROCESSED", "latency": latency}

async def enterprise_orchestrator():
    """
    Orchestrates thousands of concurrent operations on a single thread.
    Equivalent to assembling tasks and awaiting Task.WhenAll() in C#.
    """
    print("[Orchestrator] Spawning 100 concurrent network tasks on the event loop...")
    start_time = time.perf_counter()

    # 1. Create a collection of un-awaited coroutine objects (deferred execution futures)
    tasks = [fetch_api_payload(i) for i in range(1, 101)]

    # 2. Fire them all concurrently.
    # return_exceptions=True stops a single task failure from blowing up the whole batch.
    results = await asyncio.gather(*tasks, return_exceptions=True)

    # 3. Process the results stream
    success_count = 0
    failure_count = 0

    for res in results:
        if isinstance(res, Exception):
            failure_count += 1
        else:
            success_count += 1

    end_time = time.perf_counter()
    print(f"\n[Execution Summary]")
    print(f"-> Total Time Elapsed: {end_time - start_time:.4f} seconds")
    print(f"-> Successful Requests: {success_count}")
    print(f"-> Blown/Caught Exceptions: {failure_count}")

# --- Special Execution Rule for Jupyter/Google Colab Environments ---
# In a normal Python script, you start the runtime loop via: asyncio.run(enterprise_orchestrator())
# Because Google Colab is already running an active async event loop inside its kernel,
# we use top-level await directly in the cell:
await enterprise_orchestrator()


[Orchestrator] Spawning 100 concurrent network tasks on the event loop...

[Execution Summary]
-> Total Time Elapsed: 0.4893 seconds
-> Successful Requests: 88
-> Blown/Caught Exceptions: 12


In [ ]:
from numba import cuda

@cuda.jit
def gpu_matrix_addition(A, B, C):
    """
    This is written in Python, but it DOES NOT run on the CPU.
    NVIDIA's Numba compiles this directly into native GPU assembly instructions.
    """
    # Determine the absolute absolute thread position on the GPU grid
    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        # Thousands of GPU cores execute this line simultaneously in parallel
        C[row, col] = A[row, col] + B[row, col]


In [ ]:
import functools
import time

def enterprise_telemetry(func):
    """
    An Architectural Interceptor Decorator.
    Equivalent to a C# Action Interceptor or PostSharp Aspect.
    """
    # functools.wraps preserves the original function's name and metadata (__name__, __doc__).
    # Without this, the wrapped function would report its name as 'wrapper' to logging frameworks.
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # *args collects positional arguments into a tuple.
        # **kwargs collects keyword arguments into a dictionary.
        # This makes the wrapper fully generic, accepting any method signature.

        start_time = time.perf_counter()
        print(f"[Telemetry Log] Entering Service Execution Layer: '{func.__name__}'")

        try:
            # Execute the actual target function payload on the call stack
            result = func(*args, **kwargs)

            end_time = time.perf_counter()
            print(f"[Telemetry Log] Execution Completed. Duration: {end_time - start_time:.6f} seconds.")
            return result

        except Exception as error:
            print(f"[CRITICAL ERROR] Intercepted crash in '{func.__name__}': {error}")
            print("[Circuit Breaker] Tripping connection pool safety valve...")
            raise # Re-throw the exception exactly like 'throw;' in C#

    return wrapper

# --- Application Layer ---

@enterprise_telemetry
def execute_database_bulk_insert(records: list, target_table: str) -> str:
    """Simulates a core transactional business operation."""
    # Simulate processing time
    time.sleep(0.15)
    print(f"  [DB Engine] Successfully inserted {len(records)} entries into table '{target_table}'")
    return "SUCCESS_COMMIT"

# --- Test Execution Lab ---
print("Invoking decorated service pipeline...")
status = execute_database_bulk_insert([101, 102, 103], target_table="Customers")
print(f"Return payload caught by runner: {status}\n")

print("-" * 50)

# Demonstrating how the generic wrapper handles execution failures seamlessly
@enterprise_telemetry
def broken_network_call():
    raise ConnectionRefusedError("Remote cluster node dropped connection.")

try:
    broken_network_call()
except ConnectionRefusedError:
    print("[Runner Core] Exception caught at root boundary layer safely.")


Invoking decorated service pipeline...
[Telemetry Log] Entering Service Execution Layer: 'execute_database_bulk_insert'
  [DB Engine] Successfully inserted 3 entries into table 'Customers'
[Telemetry Log] Execution Completed. Duration: 0.150253 seconds.
Return payload caught by runner: SUCCESS_COMMIT

--------------------------------------------------
[Telemetry Log] Entering Service Execution Layer: 'broken_network_call'
[CRITICAL ERROR] Intercepted crash in 'broken_network_call': Remote cluster node dropped connection.
[Circuit Breaker] Tripping connection pool safety valve...
[Runner Core] Exception caught at root boundary layer safely.
